Financial metrics

In [ ]:
%pip install yfinance edgartools beautifulsoup4 requests pandas numpy

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime
import pprint
import requests
import pandas as pd
from bs4 import BeautifulSoup
import re
from edgar import *
import edgar
import os
from openpyxl import load_workbook

1) Fetching tickers

In [ ]:
def get_sp500_tickers_bs4():
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                      '(KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")
        table = soup.find("table", {"id": "constituents"})
        if not table:
            print("Could not find S&P 500 table.")
            return []

        tickers = []
        excluded = []

        rows = table.find("tbody").find_all("tr")
        for row in rows[1:]:
            cells = row.find_all("td")
            if len(cells) < 4:
                continue

            ticker = cells[0].get_text(strip=True).replace(".", "-")
            sector = cells[2].get_text(strip=True)

            if sector != "Financials":
                tickers.append(ticker)
            else:
                excluded.append(ticker)

        print(f"Retrieved {len(tickers)} non-financial tickers.")
        print(f"Excluded {len(excluded)} financial tickers.")
        return tickers

    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
        return []

if __name__ == "__main__":
    print("Fetching S&P 500 tickers from Wikipedia...")
    tickers = get_sp500_tickers_bs4()
    tickers = [t.replace('.', '-') for t in tickers]
    print(f"Retrieved {len(tickers)} tickers from Wikipedia.\n")

    for i, ticker in enumerate(tickers, 1):
        print(f"{i}. {ticker}")

2) Extracting raw financial data

In [ ]:
def get(df, col, *keys):
    if df is None or df.empty or col not in df.columns:
        return None

    idx_map = {str(idx).strip().lower(): idx for idx in df.index}

    for k in keys:
        if k in df.index:
            try:
                v = df.loc[k, col]
                if pd.notna(v):
                    return float(v)
            except Exception:
                pass

    for k in keys:
        real = idx_map.get(str(k).strip().lower())
        if real is not None:
            try:
                v = df.loc[real, col]
                if pd.notna(v):
                    return float(v)
            except Exception:
                pass

    return None


def safe_div(n, d):
    if n is None or d in [None, 0]:
        return None
    try:
        return n / d
    except Exception:
        return None


def fetch_financials_from_yfinance(ticker, years=4):
    try:
        stock = yf.Ticker(ticker)

        income = stock.income_stmt
        income_alt = stock.financials
        balance = stock.balance_sheet
        cashflow = stock.cashflow

        if income is None or income.empty:
            return []

        # Only keep these fiscal years
        valid_years = {2022, 2023, 2024, 2025}

        info = stock.info
        sector = info.get("sector")
        industry = info.get("industry")
        shares_outstanding = info.get("sharesOutstanding")
        current_market_cap = info.get("marketCap")

        results = []

        for col in income.columns:
            year = getattr(col, "year", None)

            # Only fetch 2022-2025 financial data
            if year not in valid_years:
                continue

            def g_income(col, *keys):
                v = get(income, col, *keys)
                if v is None:
                    v = get(income_alt, col, *keys)
                return v

            # Raw extracted data 
            revenue = g_income(
                col,
                "Total Revenue", "Revenue", "TotalRevenue",
                "Net Revenue", "Revenues", "Total Net Revenue",
                "Operating Revenue"
            )

            gross_profit = g_income(col, "Gross Profit", "GrossProfit", "Gross Income")
            if gross_profit is None:
                cost_of_revenue = g_income(
                    col,
                    "Cost Of Revenue", "Cost of Revenue", "CostOfRevenue",
                    "Reconciled Cost Of Revenue", "Cost Of Goods Sold",
                    "Cost Of Sales", "Reconciled Cost Of Goods Sold"
                )
                if revenue is not None and cost_of_revenue is not None:
                    gross_profit = revenue - cost_of_revenue

            ebit = g_income(
                col,
                "EBIT", "Ebit", "Operating Income", "Operating Profit",
                "Total Operating Income As Reported", "Operating Income Loss",
                "Income From Operations", "Earnings Before Interest And Taxes"
            )

            da = get(
                cashflow, col,
                "Depreciation And Amortization",
                "Depreciation Amortization Depletion",
                "Depreciation", "Depreciation And Depletion",
                "Amortization Of Intangibles",
                "Depreciation Amortization",
                "Depletion And Amortization",
                "Reconciled Depreciation"
            )

            ebitda = g_income(
                col,
                "EBITDA", "Ebitda", "Normalized EBITDA", "Adjusted EBITDA"
            )
            if ebitda is None and ebit is not None and da is not None:
                ebitda = ebit + da

            net_income = g_income(
                col,
                "Net Income",
                "Net Income Common Stockholders",
                "Net Income Including Noncontrolling Interests",
                "Net Income From Continuing Operations",
                "Net Income Applicable To Common Shares",
                "Net Income From Continuing And Discontinued Operation",
                "Diluted NI Available To Com Stockholders",
                "Diluted NI Availto Com Stockholders",
                "Net Income Continuous Operations",
                "Profit Loss",
                "Net Income From Continuing Operation Net Minority Interest"
            )

            interest_expense = g_income(
                col,
                "Interest Expense",
                "InterestExpense",
                "Interest Expense Non Operating",
                "Interest And Debt Expense",
                "Net Non Operating Interest Income Expense",
                "Interest Expense Operating",
                "Net Interest Expense",
                "Total Interest Expense"
            )

            total_assets = get(balance, col, "Total Assets", "TotalAssets", "Total Assets Net")

            shareholders_equity = get(
                balance, col,
                "Stockholders Equity", "Total Equity Gross Minority Interest",
                "Common Stock Equity", "Total Stockholder Equity",
                "Shareholders Equity", "Common Equity", "Total Equity",
                "Equity", "Net Assets"
            )

            total_liabilities = get(
                balance, col,
                "Total Liabilities Net Minority Interest",
                "Total Liabilities", "TotalLiabilities",
                "Total Liabilities Net", "Liabilities", "Total Liab",
                "Total Liabilities And Minority Interest",
                "Total Liabilities Net Of Minority Interest"
            )

            if total_liabilities is None and total_assets is not None and shareholders_equity is not None:
                total_liabilities = total_assets - shareholders_equity

            if shareholders_equity is None and total_assets is not None and total_liabilities is not None:
                shareholders_equity = total_assets - total_liabilities

            current_assets = get(
                balance, col,
                "Current Assets", "Total Current Assets",
                "CurrentAssets", "Current Assets Total",
                "Cash Cash Equivalents And Short Term Investments",
                "Cash And Cash Equivalents"
            )

            current_liabilities = get(
                balance, col,
                "Current Liabilities", "Total Current Liabilities",
                "CurrentLiabilities", "Current Liabilities Total",
                "Current Debt", "Current Debt And Capital Lease Obligation",
                "Payables And Accrued Expenses"
            )

            short_term_debt = get(
                balance, col,
                "Short Term Debt", "Current Debt",
                "Current Debt And Capital Lease Obligation",
                "Short Long Term Debt"
            )

            long_term_debt = get(
                balance, col,
                "Long Term Debt", "LongTermDebt",
                "Long Term Borrowings",
                "Long Term Debt And Capital Lease Obligation"
            )

            total_debt = get(
                balance, col,
                "Total Debt", "TotalDebt",
                "Long Term Debt And Capital Lease Obligation",
                "Total Long Term Debt", "Short Long Term Debt Total"
            )

            if total_debt is None:
                total_debt = (short_term_debt or 0) + (long_term_debt or 0)

            cash_equiv = get(
                balance, col,
                "Cash And Cash Equivalents",
                "Cash Cash Equivalents And Short Term Investments",
                "Cash Equivalents",
                "Cash And Short Term Investments",
                "Cash Financial",
                "Cash And Due From Banks",
                "Cash Items",
                "Free Cash"
            ) or 0.0

            net_debt = total_debt - cash_equiv if total_debt is not None else None

            operating_cf = get(
                cashflow, col,
                "Operating Cash Flow", "Cash From Operations",
                "Total Cash From Operating Activities",
                "Net Cash Provided By Operating Activities",
                "Cash Flows From Used In Operating Activities",
                "Net Cash From Operating Activities",
                "Cash Generated From Operations",
                "Cash Flow From Continuing Operating Activities"
            )

            capex = get(
                cashflow, col,
                "Capital Expenditure",
                "Capital Expenditures",
                "Capital Expenditure Reported",
                "Purchase Of Property Plant And Equipment",
                "Purchases Of Property And Equipment",
                "Purchase Of PPE",
                "Purchase Of Ppe",
                "Net PPE Purchase And Sale",
                "Purchase Of Business"
            )

            reported_fcf = get(cashflow, col, "Free Cash Flow")

            capex_for_fcf = min(capex, 0) if capex is not None else None
            capex_abs = abs(capex) if capex is not None else None

            if operating_cf is not None and capex_for_fcf is not None:
                fcf = operating_cf + capex_for_fcf
            else:
                fcf = None

            market_cap = current_market_cap
            enterprise_value = (
                market_cap + total_debt - cash_equiv
                if market_cap is not None and total_debt is not None
                else None
            )

            # ── Derived data ────────────────────────────────────────────
            gross_margin = safe_div(gross_profit, revenue)
            ebitda_margin = safe_div(ebitda, revenue)
            operating_margin = safe_div(ebit, revenue)
            net_margin = safe_div(net_income, revenue)

            roe = safe_div(net_income, shareholders_equity)
            roa = safe_div(net_income, total_assets)

            debt_to_equity = safe_div(total_debt, shareholders_equity)
            debt_to_ebitda = safe_div(total_debt, ebitda)
            net_debt_to_ebitda = safe_div(net_debt, ebitda)
            current_ratio = safe_div(current_assets, current_liabilities)

            ev_to_ebitda = safe_div(enterprise_value, ebitda)
            ev_to_revenue = safe_div(enterprise_value, revenue)
            price_to_earnings = safe_div(market_cap, net_income)

            fcf_margin = safe_div(fcf, revenue)
            fcf_conversion = safe_div(fcf, ebitda)
            fcf_to_debt = safe_div(fcf, total_debt)
            debt_to_fcf = safe_div(total_debt, fcf)

            interest_coverage = (
                safe_div(ebit, abs(interest_expense))
                if interest_expense not in [None, 0]
                else None
            )

            cash_interest_coverage = (
                safe_div(ebitda - capex_abs, abs(interest_expense))
                if ebitda is not None and capex_abs is not None and interest_expense not in [None, 0]
                else None
            )

            capex_to_revenue = safe_div(capex_abs, revenue)
            capex_to_ebitda = safe_div(capex_abs, ebitda)
            ocf_to_debt = safe_div(operating_cf, total_debt)

            results.append({
                "ticker": ticker,
                "year": year,
                "sector": sector,
                "industry": industry,
                "shares_outstanding": shares_outstanding,

                "market_cap": market_cap,
                "enterprise_value": enterprise_value,

                "revenue": revenue,
                "gross_profit": gross_profit,
                "ebitda": ebitda,
                "ebit": ebit,
                "net_income": net_income,
                "interest_expense": interest_expense,

                "total_assets": total_assets,
                "total_liabilities": total_liabilities,
                "shareholders_equity": shareholders_equity,
                "current_assets": current_assets,
                "current_liabilities": current_liabilities,
                "total_debt": total_debt,
                "short_term_debt": short_term_debt,
                "long_term_debt": long_term_debt,
                "cash_equiv": cash_equiv,
                "net_debt": net_debt,

                "operating_cf": operating_cf,
                "capex": capex,
                "capex_for_fcf": capex_for_fcf,
                "capex_abs": capex_abs,
                "reported_fcf": reported_fcf,
                "fcf": fcf,

                "gross_margin": gross_margin,
                "ebitda_margin": ebitda_margin,
                "operating_margin": operating_margin,
                "net_margin": net_margin,
                "roe": roe,
                "roa": roa,

                "debt_to_equity": debt_to_equity,
                "debt_to_ebitda": debt_to_ebitda,
                "net_debt_to_ebitda": net_debt_to_ebitda,
                "current_ratio": current_ratio,

                "ev_to_ebitda": ev_to_ebitda,
                "ev_to_revenue": ev_to_revenue,
                "price_to_earnings": price_to_earnings,

                "fcf_margin": fcf_margin,
                "fcf_conversion": fcf_conversion,
                "fcf_to_debt": fcf_to_debt,
                "debt_to_fcf": debt_to_fcf,

                "interest_coverage": interest_coverage,
                "cash_interest_coverage": cash_interest_coverage,

                "capex_to_revenue": capex_to_revenue,
                "capex_to_ebitda": capex_to_ebitda,

                "ocf_to_debt": ocf_to_debt,
            })

        return results

    except Exception as e:
        print(f"  ✗ Error fetching {ticker}: {e}")
        return []
        
if __name__ == "__main__":
    print("Fetching S&P 500 tickers from Wikipedia...")
    tickers = get_sp500_tickers_bs4()
    tickers = [t.replace(".", "-") for t in tickers]
    print(f"Retrieved {len(tickers)} tickers.")

    all_data, failed = [], []
    test_tickers = tickers

    for i, ticker in enumerate(test_tickers, 1):
        print(f"[{i}/{len(test_tickers)}] Fetching {ticker}...")
        rows = fetch_financials_from_yfinance(ticker)

        if rows:
            all_data.extend(rows)
        else:
            failed.append(ticker)

    if all_data:
        df = pd.DataFrame(all_data)

        preferred_cols = [
            "ticker", "year", "sector", "industry", "shares_outstanding",
            "market_cap", "enterprise_value",

            "revenue", "gross_profit", "ebitda", "ebit", "net_income", "interest_expense",
            "total_assets", "total_liabilities", "shareholders_equity",
            "current_assets", "current_liabilities",
            "total_debt", "short_term_debt", "long_term_debt", "cash_equiv", "net_debt",

            "operating_cf", "capex", "capex_for_fcf", "capex_abs", "reported_fcf", "fcf",

            "gross_margin", "ebitda_margin", "operating_margin", "net_margin",
            "roe", "roa",

            "debt_to_equity", "debt_to_ebitda", "net_debt_to_ebitda", "current_ratio",
            "ev_to_ebitda", "ev_to_revenue", "price_to_earnings",

            "fcf_margin", "fcf_conversion", "fcf_to_debt", "debt_to_fcf",
            "interest_coverage", "cash_interest_coverage",
            "capex_to_revenue", "capex_to_ebitda",
            "ocf_to_debt",
        ]

        existing_cols = [c for c in preferred_cols if c in df.columns]
        df = df[existing_cols]

        filename = "sp500_financials_output.xlsx"
        df.to_excel(filename, index=False)

        if os.path.exists(filename):
            print(f"\n✓ Saved '{filename}'  |  Rows: {len(df)}  |  Cols: {len(df.columns)}")
            print(f"  File size: {os.path.getsize(filename):,} bytes")
        else:
            print(f"\n✗ Failed to save '{filename}'.")

        check_cols = [
            "revenue", "gross_profit", "ebitda", "ebit", "net_income", "interest_expense",
            "total_assets", "total_liabilities", "shareholders_equity",
            "current_assets", "current_liabilities",
            "total_debt", "net_debt", "cash_equiv",

            "operating_cf", "capex", "capex_for_fcf", "capex_abs", "reported_fcf", "fcf",

            "market_cap", "enterprise_value",

            "gross_margin", "ebitda_margin", "operating_margin", "net_margin",
            "roe", "roa",

            "debt_to_equity", "debt_to_ebitda", "net_debt_to_ebitda", "current_ratio",
            "ev_to_ebitda", "ev_to_revenue", "price_to_earnings",

            "fcf_margin", "fcf_conversion", "fcf_to_debt", "debt_to_fcf",
            "interest_coverage", "cash_interest_coverage",
            "capex_to_revenue", "capex_to_ebitda",
            "ocf_to_debt",
        ]

        print("\n--- Missing data summary ---")
        for col in check_cols:
            if col in df.columns:
                missing_df = df[df[col].isna()]
                if len(missing_df) > 0:
                    missing_tickers = sorted(missing_df["ticker"].dropna().unique().tolist())
                    print(f"\nResulted in None: {col}")
                    print(f"Count of tickers: {len(missing_tickers)}")
                    print(f"Tickers: {missing_tickers}")

        print("\nFailed tickers:")
        print(failed)

    else:
        print("\nNo data was collected.")

3) Evaluating and Scoring data

In [ ]:
df = pd.read_excel("sp500_financials_output.xlsx")

df["original_order"] = range(len(df))
df["year"] = pd.to_numeric(df["year"], errors="coerce")


df_latest = (
    df.sort_values(["ticker", "year"])
      .groupby("ticker", as_index=False)
      .tail(1)
      .copy()
)

df_latest = df_latest.sort_values("original_order")


def score_higher_better(x, bands):
    if pd.isna(x):
        return np.nan
    for i, bound in enumerate(bands):
        if x < bound:
            return i
    return 5


def score_lower_better(x, bands):
    if pd.isna(x):
        return np.nan
    for i, bound in enumerate(bands):
        if x <= bound:
            return 5 - i
    return 0


def autofit_excel(filename):
    wb = load_workbook(filename)
    ws = wb.active

    for col in ws.columns:
        max_length = 0
        col_letter = col[0].column_letter

        for cell in col:
            if cell.value is not None:
                max_length = max(max_length, len(str(cell.value)))

        ws.column_dimensions[col_letter].width = min(max_length + 2, 45)

    wb.save(filename)


#Hard filters
critical_cols = [
    "fcf",
    "interest_coverage",
    "debt_to_ebitda",
    "current_ratio",
    "capex_to_ebitda",
]

df_latest["missing_critical_data"] = df_latest[critical_cols].isna().any(axis=1)

df_latest["pass_fcf_filter"] = df_latest["fcf"] > 0
df_latest["pass_interest_coverage_filter"] = df_latest["interest_coverage"] >= 2.0
df_latest["pass_debt_to_ebitda_filter"] = df_latest["debt_to_ebitda"] <= 5.0
df_latest["pass_current_ratio_filter"] = df_latest["current_ratio"] >= 1.0
df_latest["pass_capex_to_ebitda_filter"] = df_latest["capex_to_ebitda"] <= 0.50

df_latest["pass_hard_filters"] = (
    (~df_latest["missing_critical_data"]) &
    df_latest["pass_fcf_filter"] &
    df_latest["pass_interest_coverage_filter"] &
    df_latest["pass_debt_to_ebitda_filter"] &
    df_latest["pass_current_ratio_filter"] &
    df_latest["pass_capex_to_ebitda_filter"]
)


#Scoring 
df_latest["fcf_score"] = df_latest["fcf"].apply(
    lambda x: score_higher_better(x, [0, 100_000_000, 500_000_000, 1_000_000_000, 5_000_000_000])
)

df_latest["fcf_margin_score"] = df_latest["fcf_margin"].apply(
    lambda x: score_higher_better(x, [0, 0.05, 0.10, 0.15, 0.20])
)

df_latest["fcf_conversion_score"] = df_latest["fcf_conversion"].apply(
    lambda x: score_higher_better(x, [0, 0.20, 0.40, 0.60, 0.80])
)

df_latest["fcf_to_debt_score"] = df_latest["fcf_to_debt"].apply(
    lambda x: score_higher_better(x, [0, 0.05, 0.10, 0.15, 0.25])
)

df_latest["debt_to_ebitda_score"] = df_latest["debt_to_ebitda"].apply(
    lambda x: score_lower_better(x, [1, 2, 3, 4, 5])
)

df_latest["net_debt_to_ebitda_score"] = df_latest["net_debt_to_ebitda"].apply(
    lambda x: score_lower_better(x, [1, 2, 3, 4, 5])
)

df_latest["interest_coverage_score"] = df_latest["interest_coverage"].apply(
    lambda x: score_higher_better(x, [1, 2, 3, 5, 8])
)

df_latest["cash_interest_coverage_score"] = df_latest["cash_interest_coverage"].apply(
    lambda x: score_higher_better(x, [1, 2, 3, 5, 8])
)

df_latest["capex_to_revenue_score"] = df_latest["capex_to_revenue"].apply(
    lambda x: score_lower_better(x, [0.02, 0.05, 0.10, 0.15, 0.25])
)

df_latest["capex_to_ebitda_score"] = df_latest["capex_to_ebitda"].apply(
    lambda x: score_lower_better(x, [0.05, 0.10, 0.20, 0.35, 0.50])
)


#Section scores 
df_latest["cash_flow_strength_score"] = df_latest[
    ["fcf_score", "fcf_margin_score", "fcf_conversion_score"]
].mean(axis=1)

df_latest["deleveraging_capacity_score"] = df_latest[
    ["fcf_to_debt_score"]
].mean(axis=1)

df_latest["leverage_score"] = df_latest[
    ["debt_to_ebitda_score", "net_debt_to_ebitda_score"]
].mean(axis=1)

df_latest["coverage_score"] = df_latest[
    ["interest_coverage_score", "cash_interest_coverage_score"]
].mean(axis=1)

df_latest["capex_burden_score"] = df_latest[
    ["capex_to_revenue_score", "capex_to_ebitda_score"]
].mean(axis=1)


#Final score 
df_latest["lbo_financial_score"] = (
    0.30 * df_latest["cash_flow_strength_score"] +
    0.25 * df_latest["deleveraging_capacity_score"] +
    0.15 * df_latest["leverage_score"] +
    0.15 * df_latest["coverage_score"] +
    0.15 * df_latest["capex_burden_score"]
) * 20


#Rating + Status 
def assign_rating(row):
    score = row["lbo_financial_score"]

    if row["missing_critical_data"] or pd.isna(score):
        return "Insufficient data"

    if score < 70:
        return "Marginal"

    if row["pass_hard_filters"] and score >= 85:
        return "Ideal"

    if row["pass_hard_filters"] and score >= 70:
        return "Attractive"

    if (not row["pass_hard_filters"]) and score >= 85:
        return "Situational"

    return "Marginal"


df_latest["lbo_rating"] = df_latest.apply(assign_rating, axis=1)

df_latest["lbo_status"] = df_latest["pass_hard_filters"].map({
    True: "Pass",
    False: "Fail"
})



if "original_order" in df_latest.columns:
    df_latest = df_latest.drop(columns=["original_order"])

def add_gap(df, name):
    df[name] = ""
    return name


ordered_cols = [
    "ticker", "year", "sector",
    "missing_critical_data", add_gap(df_latest, "gap_1"),

    "pass_fcf_filter", "pass_interest_coverage_filter", "pass_debt_to_ebitda_filter",
    "pass_current_ratio_filter", "pass_capex_to_ebitda_filter", "pass_hard_filters",
    add_gap(df_latest, "gap_2"),

    "fcf", "fcf_margin", "fcf_conversion", "fcf_score", "fcf_margin_score",
    "fcf_conversion_score", "cash_flow_strength_score",
    add_gap(df_latest, "gap_3"),

    "fcf_to_debt", "fcf_to_debt_score", "deleveraging_capacity_score",
    add_gap(df_latest, "gap_4"),

    "debt_to_ebitda", "net_debt_to_ebitda", "debt_to_ebitda_score",
    "net_debt_to_ebitda_score", "leverage_score",
    add_gap(df_latest, "gap_5"),

    "interest_coverage", "cash_interest_coverage", "interest_coverage_score",
    "cash_interest_coverage_score", "coverage_score",
    add_gap(df_latest, "gap_6"),

    "capex_to_revenue", "capex_to_ebitda", "capex_to_revenue_score",
    "capex_to_ebitda_score", "capex_burden_score",
    add_gap(df_latest, "gap_7"),

    "lbo_financial_score", "lbo_status", "lbo_rating"
]

ordered_cols = [c for c in ordered_cols if c in df_latest.columns]

df_clean = df_latest[ordered_cols].copy()

df_clean.columns = [
    "" if str(c).startswith("gap_") else c
    for c in df_clean.columns
]


rating_order = {
    "Ideal": 1,
    "Attractive": 2,
    "Situational": 3,
    "Marginal": 4,
    "Insufficient data": 5
}

df_ranked = df_clean.copy()

df_ranked["_rating_order"] = df_ranked["lbo_rating"].map(rating_order)

df_ranked = df_ranked.sort_values(
    ["_rating_order", "lbo_financial_score"],
    ascending=[True, False]
).drop(columns=["_rating_order"])

df_ranked.insert(0, "lbo_rank", range(1, len(df_ranked) + 1))

df_ranked.to_excel("lbo_finscore.xlsx", index=False)
autofit_excel("lbo_finscore.xlsx")